<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/CH5_Cognitive_Architectures_with_LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# Groq API 키 설정
YOUR_API_KEY = userdata.get('groq')
os.environ["GROQ_API_KEY"] = YOUR_API_KEY

In [2]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.2 MB/s eta 0:00:00


In [6]:
# @title **Architecture #1: LLM Call**

from langchain_groq import ChatGroq
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

class State(TypedDict):
    # Messages have the type "list". The `add_messages`
    # function in the annotation defines how this state should
    # be updated (in this case, it appends new messages to the
    # list, rather than replacing the previous messages)
    messages: Annotated[list, add_messages]

def chatbot(state: State):
    answer = model.invoke(state["messages"])
    return {"messages": [answer]}

builder = StateGraph(State)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()

# Example usage

input = {"messages": [HumanMessage("hi!")]}
for chunk in graph.stream(input):
    print(chunk)

{'chatbot': {'messages': [AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 37, 'total_tokens': 60, 'completion_time': 0.043344321, 'completion_tokens_details': None, 'prompt_time': 0.003200298, 'prompt_tokens_details': None, 'queue_time': 0.221307436, 'total_time': 0.046544619}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f84bc-17f1-7a60-aa8a-a74e63678b9f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 23, 'total_tokens': 60})]}}


In [5]:
[chunk for chunk in graph.stream(input)]

[{'chatbot': {'messages': [AIMessage(content="It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 37, 'total_tokens': 60, 'completion_time': 0.045307526, 'completion_tokens_details': None, 'prompt_time': 0.001911826, 'prompt_tokens_details': None, 'queue_time': 0.008389506, 'total_time': 0.047219352}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f84ba-67b5-79d0-aaef-ec58ac42d2b4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 23, 'total_tokens': 60})]}}]

In [15]:
# @title **Architecture #2: Chain**

from typing import Annotated, TypedDict
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# useful to generate SQL query
model_low_temp = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)
# useful to generate natural language outputs
model_high_temp = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)

class State(TypedDict):
    # to track conversation history
    messages: Annotated[list, add_messages]
    # input
    user_query: str
    # output
    sql_query: str
    sql_explanation: str

class Input(TypedDict):
    user_query: str

class Output(TypedDict):
    sql_query: str
    sql_explanation: str

generate_prompt = SystemMessage(
    "You are a helpful data analyst, who generates SQL queries for users based on their questions."
)

def generate_sql(state: State) -> State:
    user_message = HumanMessage(state["user_query"])
    messages = [generate_prompt, *state["messages"], user_message]
    res = model_low_temp.invoke(messages)
    return {
        "sql_query": res.content,
        # update conversation history
        "messages": [user_message, res],
    }

explain_prompt = SystemMessage(
    "You are a helpful data analyst, who explains SQL queries to users."
)

def explain_sql(state: State) -> State:
    messages = [
        explain_prompt,
        # contains user's query and SQL query from prev step
        *state["messages"],
    ]
    res = model_high_temp.invoke(messages)
    return {
        "sql_explanation": res.content,
        # update conversation history
        "messages": res,
    }

builder = StateGraph(State, input=Input, output=Output)
builder.add_node("generate_sql", generate_sql)
builder.add_node("explain_sql", explain_sql)
builder.add_edge(START, "generate_sql")
builder.add_edge("generate_sql", "explain_sql")
builder.add_edge("explain_sql", END)

graph = builder.compile()

# Example usage
result = graph.invoke({"user_query": "What is the total sales for each product?"})
print(result)

/tmp/ipykernel_2806/4178124024.py:61: LangGraphDeprecatedSinceV05: `input` is deprecated and will be removed. Please use `input_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  builder = StateGraph(State, input=Input, output=Output)
/tmp/ipykernel_2806/4178124024.py:61: LangGraphDeprecatedSinceV05: `output` is deprecated and will be removed. Please use `output_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  builder = StateGraph(State, input=Input, output=Output)


{'sql_query': '**SQL Query: Total Sales per Product**\n\nTo calculate the total sales for each product, you can use the following SQL query:\n\n```sql\nSELECT \n    p.product_name, \n    SUM(o.order_total) AS total_sales\nFROM \n    orders o\nJOIN \n    products p ON o.product_id = p.product_id\nGROUP BY \n    p.product_name\nORDER BY \n    total_sales DESC;\n```\n\n**Assumptions:**\n\n* `orders` table contains the order details with `order_total` column representing the total sales amount for each order.\n* `products` table contains the product information with `product_name` and `product_id` columns.\n* `product_id` is the common column between `orders` and `products` tables to join them.\n\n**How it works:**\n\n1. The query joins the `orders` and `products` tables based on the `product_id` column.\n2. It groups the results by the `product_name` column.\n3. For each group, it calculates the total sales using the `SUM` aggregation function on the `order_total` column.\n4. The results 

In [16]:
print(result['sql_query'])

**SQL Query: Total Sales per Product**

To calculate the total sales for each product, you can use the following SQL query:

```sql
SELECT 
    p.product_name, 
    SUM(o.order_total) AS total_sales
FROM 
    orders o
JOIN 
    products p ON o.product_id = p.product_id
GROUP BY 
    p.product_name
ORDER BY 
    total_sales DESC;
```

**Assumptions:**

* `orders` table contains the order details with `order_total` column representing the total sales amount for each order.
* `products` table contains the product information with `product_name` and `product_id` columns.
* `product_id` is the common column between `orders` and `products` tables to join them.

**How it works:**

1. The query joins the `orders` and `products` tables based on the `product_id` column.
2. It groups the results by the `product_name` column.
3. For each group, it calculates the total sales using the `SUM` aggregation function on the `order_total` column.
4. The results are sorted in descending order by the `total

In [17]:
print(result['sql_explanation'])

In [19]:
pip install -qU langchain-huggingface

In [20]:
# @title **Architecture #3: Router**

from typing import Annotated, Literal, TypedDict
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.vectorstores.in_memory import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# HuggingFace 384차원 임베딩 모델 설정
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# useful to generate SQL query
model_low_temp = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)
# useful to generate natural language outputs
model_low_temp = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)

class State(TypedDict):
    # to track conversation history
    messages: Annotated[list, add_messages]
    # input
    user_query: str
    # output
    domain: Literal["records", "insurance"]
    documents: list[Document]
    answer: str

class Input(TypedDict):
    user_query: str

class Output(TypedDict):
    documents: list[Document]
    answer: str

# Sample documents for testing
sample_docs = [
    Document(page_content="Patient medical record...", metadata={"domain": "records"}),
    Document(
        page_content="Insurance policy details...", metadata={"domain": "insurance"}
    ),
]

# Initialize vector stores
medical_records_store = InMemoryVectorStore.from_documents(sample_docs, embeddings)
medical_records_retriever = medical_records_store.as_retriever()

insurance_faqs_store = InMemoryVectorStore.from_documents(sample_docs, embeddings)
insurance_faqs_retriever = insurance_faqs_store.as_retriever()

router_prompt = SystemMessage(
    """You need to decide which domain to route the user query to. You have two domains to choose from:
- records: contains medical records of the patient, such as diagnosis, treatment, and prescriptions.
- insurance: contains frequently asked questions about insurance policies, claims, and coverage.

Output only the domain name."""
)

def router_node(state: State) -> State:
    user_message = HumanMessage(state["user_query"])
    messages = [router_prompt, *state["messages"], user_message]
    res = model_low_temp.invoke(messages)
    return {
        "domain": res.content,
        # update conversation history
        "messages": [user_message, res],
    }

def pick_retriever(
    state: State,
) -> Literal["retrieve_medical_records", "retrieve_insurance_faqs"]:
    if state["domain"] == "records":
        return "retrieve_medical_records"
    else:
        return "retrieve_insurance_faqs"

def retrieve_medical_records(state: State) -> State:
    documents = medical_records_retriever.invoke(state["user_query"])
    return {
        "documents": documents,
    }

def retrieve_insurance_faqs(state: State) -> State:
    documents = insurance_faqs_retriever.invoke(state["user_query"])
    return {
        "documents": documents,
    }

medical_records_prompt = SystemMessage(
    "You are a helpful medical chatbot, who answers questions based on the patient's medical records, such as diagnosis, treatment, and prescriptions."
)

insurance_faqs_prompt = SystemMessage(
    "You are a helpful medical insurance chatbot, who answers frequently asked questions about insurance policies, claims, and coverage."
)

def generate_answer(state: State) -> State:
    if state["domain"] == "records":
        prompt = medical_records_prompt
    else:
        prompt = insurance_faqs_prompt
    messages = [
        prompt,
        *state["messages"],
        HumanMessage(f"Documents: {state['documents']}"),
    ]
    res = model_high_temp.invoke(messages)
    return {
        "answer": res.content,
        # update conversation history
        "messages": res,
    }

builder = StateGraph(State, input=Input, output=Output)
builder.add_node("router", router_node)
builder.add_node("retrieve_medical_records", retrieve_medical_records)
builder.add_node("retrieve_insurance_faqs", retrieve_insurance_faqs)
builder.add_node("generate_answer", generate_answer)
builder.add_edge(START, "router")
builder.add_conditional_edges("router", pick_retriever)
builder.add_edge("retrieve_medical_records", "generate_answer")
builder.add_edge("retrieve_insurance_faqs", "generate_answer")
builder.add_edge("generate_answer", END)

graph = builder.compile()

# Example usage
input = {"user_query": "Am I covered for COVID-19 treatment?"}
for chunk in graph.stream(input):
    print(chunk)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_2806/3148439567.py:115: LangGraphDeprecatedSinceV05: `input` is deprecated and will be removed. Please use `input_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  builder = StateGraph(State, input=Input, output=Output)
/tmp/ipykernel_2806/3148439567.py:115: LangGraphDeprecatedSinceV05: `output` is deprecated and will be removed. Please use `output_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  builder = StateGraph(State, input=Input, output=Output)


{'router': {'domain': 'insurance', 'messages': [HumanMessage(content='Am I covered for COVID-19 treatment?', additional_kwargs={}, response_metadata={}, id='1663bb82-4cf6-497a-a5d5-509e69b03e0f'), AIMessage(content='insurance', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 106, 'total_tokens': 108, 'completion_time': 0.014392605, 'completion_tokens_details': None, 'prompt_time': 0.01125162, 'prompt_tokens_details': None, 'queue_time': 0.009072751, 'total_time': 0.025644225}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f84d1-22c0-7f90-a955-cacdbcc6628e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 106, 'output_tokens': 2, 'total_tokens': 108})]}}
{'retrieve_insurance_faqs': {'documents': [Document(id='26bb970e-b3ab-4ca2-9f77-64b196b7567a', metadata={'domain': '

In [21]:
input = {"user_query": "Am I covered for COVID-19 treatment?"}
for chunk in graph.stream(input):
    print(chunk)

{'router': {'domain': 'insurance', 'messages': [HumanMessage(content='Am I covered for COVID-19 treatment?', additional_kwargs={}, response_metadata={}, id='114be1a6-189b-4277-bbc0-0c2c5ab2de5f'), AIMessage(content='insurance', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 2, 'prompt_tokens': 106, 'total_tokens': 108, 'completion_time': 0.014423592, 'completion_tokens_details': None, 'prompt_time': 0.007935383, 'prompt_tokens_details': None, 'queue_time': 0.175526815, 'total_time': 0.022358975}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f84d1-dc61-7df2-b5d4-19934c1ad129-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 106, 'output_tokens': 2, 'total_tokens': 108})]}}
{'retrieve_insurance_faqs': {'documents': [Document(id='26bb970e-b3ab-4ca2-9f77-64b196b7567a', metadata={'domain': 